# Evaluation: retrieval, answers, citations and abstention

This notebook measures the **current** pipeline on a small labelled dataset. It reuses the production
functions (`load_pdf`, `index_document`, `retrieve_passages`, `answer_question`); `evaluation/` only
adds the dataset, the metrics, and loops that call them.

**Read this first**

- The corpus is a **synthetic**, 10-page report about a fictional company (`evaluation/corpus.py`). The 27
  questions were written by the same author as the corpus and nothing was tuned on them, so this is an
  *exploratory* check, not a held-out benchmark. It says little about arbitrary real PDFs.
- Cells 1-6 (corpus, index, retrieval) run locally: **no API calls**. The local embedding model is
  downloaded on first use.
- The generation section calls the **Groq API** and sends the questions and retrieved passages (synthetic text)
  to Groq. It runs only if the environment variable `EVAL_RUN_LIVE=1` and `GROQ_API_KEY` are set. Credentials
  are never read or shown in this notebook.

Run from the repository root (the notebook finds the root itself):

```powershell
# retrieval only, no API calls
uv run --group notebook jupyter execute notebooks/evaluation.ipynb --output executed.ipynb
# including live generation
$env:EVAL_RUN_LIVE = "1"; uv run --env-file .env --group notebook jupyter execute notebooks/evaluation.ipynb --output executed.ipynb
```

In [ ]:
import json, os, shutil, sys, tempfile
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))  # makes the top-level `evaluation` package importable

from IPython.display import Markdown, display

from evaluation.corpus import PAGES, build_pdf
from evaluation.dataset import load_items
from evaluation.harness import (markdown_table, run_generation, run_retrieval,
                                summarize_generation, summarize_retrieval)
from intelligent_document_assistant.ingestion import PAGE_SEPARATOR, load_pdf
from intelligent_document_assistant.retrieval import create_embed_model, index_document

TOP_K = 5          # what the app uses by default
KS = (1, 3, 5)     # cut-offs for hit rate and coverage
RUN_LIVE = os.environ.get("EVAL_RUN_LIVE") == "1"

def fmt(value):
    return "-" if value is None else f"{value:.2f}"

print("Project root:", ROOT, "| live generation enabled:", RUN_LIVE)

## 1. Load the corpus and inspect the page-aware text

The PDF is generated from `evaluation/corpus.py` and read back through the production `load_pdf`, so page labels
in the dataset refer to what the pipeline actually extracts.

In [ ]:
workdir = Path(tempfile.mkdtemp(prefix="eval_corpus_"))
document = load_pdf(build_pdf(workdir))
shutil.rmtree(workdir)  # the document is in memory; leave no PDF behind
pages = document.text.split(PAGE_SEPARATOR)
display(Markdown(f"`{document.metadata}`"))
display(Markdown(markdown_table(
    [{"page": i, "words": len(t.split()), "starts with": t[:60] + "..."} for i, t in enumerate(pages, 1)],
    ["page", "words", "starts with"])))

## 2. The labelled dataset

Each question records whether it is answerable, which page(s) hold the answer, and a rubric: a list of groups,
where every group needs at least one alternative to appear in the answer (case-insensitive substring).
`tests/test_evaluation_dataset.py` checks every label against the extracted page text.

In [ ]:
items = load_items()
by_category = {}
for it in items:
    by_category.setdefault(it.category, []).append(it)
display(Markdown(markdown_table(
    [{"category": c, "questions": len(v), "answerable": sum(i.answerable for i in v)} for c, v in by_category.items()],
    ["category", "questions", "answerable"])))
display(Markdown(markdown_table(
    [{"id": i.id, "category": i.category, "question": i.question, "answerable": i.answerable,
      "relevant pages": list(i.relevant_pages) or None} for i in items],
    ["id", "category", "question", "answerable", "relevant pages"])))

## 3. Build the index (local, no API calls)

In [ ]:
embed_model = create_embed_model()
index = index_document(document, embed_model)
nodes = list(index.docstore.docs.values())
chunks_per_page = {}
for n in nodes:
    chunks_per_page[n.metadata["page_number"]] = chunks_per_page.get(n.metadata["page_number"], 0) + 1
print(f"{len(nodes)} chunks over {len(pages)} pages; chunks per page: {dict(sorted(chunks_per_page.items()))}")

## 4. Retrieval quality (no API calls)

**Unit of relevance: the page.** Retrieval returns chunks; a page counts once however many of its chunks come back.
Only answerable questions are scored.

- **Hit@k**: at least one relevant page appears among the pages of the top-k chunks.
- **Full coverage@k**: *all* required pages appear (the stricter test for multi-page questions); mean coverage is
  the average fraction of required pages found.
- **MRR**: mean of 1/rank of the first chunk that lies on a relevant page (rank counts chunks, 0 if none retrieved).

Similarity scores are not used: they rank passages for one query and are not calibrated probabilities.

In [ ]:
retrieval_rows = run_retrieval(index, items, top_k=TOP_K, ks=KS)
overall = summarize_retrieval(retrieval_rows, KS)
summary_rows = [{"slice": "all answerable", **overall}]
for c in by_category:
    if any(r["answerable"] for r in retrieval_rows if r["category"] == c):
        summary_rows.append({"slice": c, **summarize_retrieval([r for r in retrieval_rows if r["category"] == c], KS)})
cols = ["slice", "n"] + [f"hit@{k}" for k in KS] + [f"full_coverage@{k}" for k in KS] + ["mrr"]
display(Markdown(markdown_table(
    [{k: (fmt(v) if isinstance(v, float) else v) for k, v in r.items()} for r in summary_rows], cols)))

### Ranking inspection

`retrieved chunk pages` is the page of each of the top-5 chunks, best first (repeats = several chunks from one page).

In [ ]:
display(Markdown(markdown_table(
    [{"id": r["id"], "question": r["question"], "relevant": r["relevant_pages"], "retrieved chunk pages": r["chunk_pages"],
      "hit@1": r["hit@1"], "hit@3": r["hit@3"], "hit@5": r["hit@5"], "full@5": r["coverage@5"] == 1.0, "rr": fmt(r["rr"])}
     for r in retrieval_rows if r["answerable"]],
    ["id", "question", "relevant", "retrieved chunk pages", "hit@1", "hit@3", "hit@5", "full@5", "rr"])))

misses = [r for r in retrieval_rows if r["answerable"] and r["coverage@5"] < 1.0]
display(Markdown("**Questions where not every required page was in the top 5 chunks:** " +
                 (", ".join(f"`{r['id']}` (needs {r['relevant_pages']}, got {r['chunk_pages']})" for r in misses) or "none")))

In [ ]:
display(Markdown("**Unanswerable questions: the nearest chunks are still returned** (no metric; shown for transparency)"))
display(Markdown(markdown_table(
    [{"id": r["id"], "question": r["question"], "retrieved chunk pages": r["chunk_pages"]}
     for r in retrieval_rows if not r["answerable"]],
    ["id", "question", "retrieved chunk pages"])))

## 5. Answer generation, citations and abstention (**calls the Groq API**)

Generation is scored separately from retrieval, so each failure can be assigned to a stage:
`evidence_retrieved` says whether every required page was among the retrieved pages. An answerable question that
fails although the evidence was retrieved is a **generation** failure; if the evidence was missing it is a
**retrieval** failure.

- **Answer correctness**: the rubric (a floor: it cannot judge meaning) plus manual review of the answers listed below.
- **Citation, reference level**: the cited pages include a required page. This is *not* semantic support: a valid
  citation only means the model named a passage it was shown.
- **Citation, content proxy**: the rubric facts occur in the text of the cited passages. Still a proxy, so read the answers.
- **Abstention**: positive class = *abstained*. Unanswerable and abstained = true positive; answerable but abstained
  = false positive; unanswerable but answered = false negative.

Model output is not fully deterministic even at temperature 0, so one run is one sample.

In [ ]:
generation_rows, run_info = [], None
if RUN_LIVE:
    from intelligent_document_assistant.generation import create_chat_model
    chat_model = create_chat_model()
    run_info = {"date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                "chat_model": getattr(chat_model, "model_name", type(chat_model).__name__),
                "embedding_model": embed_model.model_name, "top_k": TOP_K}
    generation_rows = run_generation(index, items, chat_model, top_k=TOP_K)
    print("Live run:", run_info)
else:
    print("Skipped: set EVAL_RUN_LIVE=1 (and GROQ_API_KEY) to run the generation evaluation. No metrics are reported without it.")

In [ ]:
if generation_rows:
    summary = summarize_generation(generation_rows)
    c = summary["confusion"]
    display(Markdown(markdown_table([
        {"": "unanswerable", "abstained": f"{c['tp']} (TP)", "answered": f"{c['fn']} (FN: invented an answer)"},
        {"": "answerable", "abstained": f"{c['fp']} (FP: refused a good question)", "answered": f"{c['tn']} (TN)"}],
        ["", "abstained", "answered"])))
    display(Markdown(f"Abstention precision = {fmt(c['precision'])}, recall = {fmt(c['recall'])} "
                     f"(counts above; only {c['tp'] + c['fn']} unanswerable questions, so treat the ratios as indicative). "
                     f"Calls that raised an error: {summary['errors']}."))
    display(Markdown(markdown_table([{
        "answerable answered": summary["answerable_answered"],
        "rubric pass": summary["rubric_pass"],
        "cited page includes a required page": summary["cited_includes_relevant"],
        "cited text contains the rubric facts": summary["cited_text_supports"],
        "abstained although evidence retrieved": summary["answerable_wrongly_abstained_with_evidence"],
        "abstained, evidence not retrieved": summary["answerable_wrongly_abstained_without_evidence"]}],
        ["answerable answered", "rubric pass", "cited page includes a required page",
         "cited text contains the rubric facts", "abstained although evidence retrieved", "abstained, evidence not retrieved"])))

In [ ]:
if generation_rows:
    display(Markdown(markdown_table(
        [{"id": r["id"], "category": r["category"], "answerable": r["answerable"], "abstained": r["abstained"],
          "rubric": r["rubric_pass"], "relevant": r["relevant_pages"] or None, "cited": r["cited_pages"] or None,
          "cited incl. relevant": r["cited_includes_relevant"], "cited text has facts": r["cited_text_supports"],
          "evidence retrieved": r["evidence_retrieved"], "error": r["error"]} for r in generation_rows],
        ["id", "category", "answerable", "abstained", "rubric", "relevant", "cited", "cited incl. relevant",
         "cited text has facts", "evidence retrieved", "error"])))

### Answers for manual review

Read each answer against the expected answer and the page text. The rubric can pass a wrong answer that happens to
contain the keywords, or fail a correct paraphrase.

In [ ]:
if generation_rows:
    expected = {i.id: i.expected_answer for i in items}
    display(Markdown(markdown_table(
        [{"id": r["id"], "question": r["question"], "expected": expected[r["id"]],
          "model answer": (r["answer"] or f"ERROR {r['error']}")[:220], "abstained": r["abstained"]} for r in generation_rows],
        ["id", "question", "expected", "model answer", "abstained"])))

## 6. Save the raw results (git-ignored)

In [ ]:
results_dir = ROOT / "evaluation" / "results"
results_dir.mkdir(exist_ok=True)
(results_dir / "latest.json").write_text(json.dumps({
    "retrieval": {"top_k": TOP_K, "ks": list(KS), "summary": overall, "rows": retrieval_rows},
    "generation": {"run": run_info, "summary": summarize_generation(generation_rows) if generation_rows else None,
                   "rows": generation_rows}}, indent=2), encoding="utf-8")
print("Saved", results_dir / "latest.json")

## Limitations

- **Synthetic, small, exploratory.** 27 questions (18 answerable, 9 unanswerable) over one 10-page document the
  author also wrote; question wording often shares words with the text, which flatters retrieval.
- No tuning was done on these questions and no chunking, embedding or prompt setting was changed in response to
  them. Any weakness seen here should be examined in a separate, controlled experiment.
- Page-level relevance ignores *where on the page* the evidence sits and whether a retrieved chunk actually contains it.
- The rubric is a keyword floor, citation checks are reference- or keyword-level, and the manual review of answers is
  by a single reviewer (not independent).
- One generation run against one model on one date; results can vary between runs and models.
- Percentages from 9 unanswerable questions are unstable; use the counts.